##GROUP NAME: DIVERSE 5
#Team Members
1. Vadlamuru varshitha krishnaveni- B24ME050
2. Malraju Srinayana- B24CS024
3. Nagineni Thanmaie- B24EC019
4. Badagoni Sri Sahasra- B24DS003
5. Pulipati Parinitha- B24CS037



# Problem Statement 3: Robot Strategy Battle

Two autonomous robots compete to collect energy cells on a warehouse grid.

- **MAX Robot = A**
- **MIN Robot = B**
- **Energy Cell = E**
- **Obstacle = #**
- **Empty Cell = .**


## Cell 1: Import Libraries

We use:

- `math` for positive and negative infinity.
- `time` to measure algorithm execution time.

In [1]:
import math
import time

## Cell 2: Define Movement Directions

The robots can move in four directions:

**UP → RIGHT → DOWN → LEFT**

This is also the normal move ordering required in the problem statement.

In [2]:
DIRECTIONS = [
    ("UP", -1, 0),
    ("RIGHT", 0, 1),
    ("DOWN", 1, 0),
    ("LEFT", 0, -1)
]

## Cell 3: Valid Position Function

This function checks whether a robot can move to a position.

A valid position must:

- Be inside the board.
- Not contain an obstacle `#`.

In [3]:
def valid_position(r, c, board, R, C):

    if r < 0 or r >= R:
        return False

    if c < 0 or c >= C:
        return False

    if board[r][c] == '#':
        return False

    return True

## Cell 4: Manhattan Distance

Because robots move only horizontally and vertically, Manhattan Distance is used:

**Distance = |r₁ − r₂| + |c₁ − c₂|**

In [4]:
def distance(p1, p2):

    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

## Cell 5: Positional Advantage

The positional advantage compares how close MAX and MIN are to the nearest remaining energy cell.

A positive value favors MAX because MIN is farther away than MAX.

In [5]:
def positional_advantage(max_pos, min_pos, energy):

    if len(energy) == 0:
        return 0

    max_distance = min(
        distance(max_pos, e)
        for e in energy
    )

    min_distance = min(
        distance(min_pos, e)
        for e in energy
    )

    return min_distance - max_distance

## Cell 6: Evaluation Function

The required evaluation is:

**Evaluation = (MAX Score − MIN Score) + Positional Advantage**

A higher evaluation value is better for MAX.

In [6]:
def evaluate(state):

    (
        max_pos,
        min_pos,
        energy,
        max_score,
        min_score,
        player
    ) = state

    score_difference = max_score - min_score

    position_difference = positional_advantage(
        max_pos,
        min_pos,
        energy
    )

    return score_difference + position_difference

## Cell 7: Successor Function

For every valid movement, this function creates a new game state.

If a robot reaches an energy cell:

- It receives **10 points**.
- That energy cell is removed.
- The next player's turn begins.

For heuristic ordering, moves collecting energy are placed first.

In [7]:
def successors(state, board, R, C, heuristic_order=False):

    (
        max_pos,
        min_pos,
        energy,
        max_score,
        min_score,
        player
    ) = state

    possible_moves = []

    for direction, dr, dc in DIRECTIONS:

        current_pos = max_pos if player == "MAX" else min_pos

        nr = current_pos[0] + dr
        nc = current_pos[1] + dc

        if not valid_position(nr, nc, board, R, C):
            continue

        new_pos = (nr, nc)
        new_energy = set(energy)

        new_max_score = max_score
        new_min_score = min_score

        collected = False

        if player == "MAX":

            if new_pos in new_energy:
                new_max_score += 10
                new_energy.remove(new_pos)
                collected = True

            new_state = (
                new_pos,
                min_pos,
                frozenset(new_energy),
                new_max_score,
                new_min_score,
                "MIN"
            )

        else:

            if new_pos in new_energy:
                new_min_score += 10
                new_energy.remove(new_pos)
                collected = True

            new_state = (
                max_pos,
                new_pos,
                frozenset(new_energy),
                new_max_score,
                new_min_score,
                "MAX"
            )

        possible_moves.append(
            (direction, new_state, collected)
        )

    if heuristic_order:
        possible_moves.sort(
            key=lambda x: x[2],
            reverse=True
        )

    return possible_moves

## Cell 8: Terminal Condition

The search stops when:

1. The specified depth limit is reached, or
2. All energy cells have been collected.

In [8]:
def terminal(state, remaining_depth):

    if remaining_depth <= 0:
        return True

    if len(state[2]) == 0:
        return True

    return False

## Cell 9: Minimax Algorithm

Minimax assumes both robots play optimally.

- MAX selects the maximum evaluation.
- MIN selects the minimum evaluation.

The algorithm explores the game tree until the terminal condition is reached.

In [9]:
def minimax(state, depth, board, R, C, statistics):

    statistics["expanded"] += 1

    if terminal(state, depth):
        return evaluate(state)

    player = state[5]

    children = successors(
        state,
        board,
        R,
        C,
        heuristic_order=False
    )

    statistics["generated"] += len(children)

    # Handle a position with no legal moves
    if not children:
        return evaluate(state)

    if player == "MAX":

        best_value = -math.inf

        for move, child, collected in children:

            value = minimax(
                child,
                depth - 1,
                board,
                R,
                C,
                statistics
            )

            best_value = max(best_value, value)

        return best_value

    else:

        best_value = math.inf

        for move, child, collected in children:

            value = minimax(
                child,
                depth - 1,
                board,
                R,
                C,
                statistics
            )

            best_value = min(best_value, value)

        return best_value

## Cell 10: Find the Best Move Using Minimax

This function checks every possible first move for MAX and selects the move with the highest Minimax value.

It also records:

- Nodes generated
- Nodes expanded
- Execution time

In [10]:
def find_best_minimax(initial_state, D, board, R, C):

    statistics = {
        "generated": 0,
        "expanded": 0
    }

    start_time = time.perf_counter()

    children = successors(
        initial_state,
        board,
        R,
        C
    )

    statistics["generated"] += len(children)

    best_value = -math.inf
    best_move = None

    for move, child, collected in children:

        value = minimax(
            child,
            D - 1,
            board,
            R,
            C,
            statistics
        )

        if value > best_value:
            best_value = value
            best_move = move

    execution_time = time.perf_counter() - start_time

    return (
        best_move,
        best_value,
        statistics["generated"],
        statistics["expanded"],
        execution_time
    )

## Cell 11: Alpha-Beta Pruning

Alpha-Beta improves Minimax by avoiding branches that cannot affect the final decision.

- **α (alpha)** = best value guaranteed for MAX.
- **β (beta)** = best value guaranteed for MIN.

A branch is pruned when:

**α ≥ β**

In [11]:
def alpha_beta(
    state,
    depth,
    alpha,
    beta,
    board,
    R,
    C,
    statistics,
    heuristic_order=False
):

    statistics["expanded"] += 1

    if terminal(state, depth):
        return evaluate(state)

    player = state[5]

    children = successors(
        state,
        board,
        R,
        C,
        heuristic_order
    )

    statistics["generated"] += len(children)

    if not children:
        return evaluate(state)

    if player == "MAX":

        value = -math.inf

        for move, child, collected in children:

            value = max(
                value,
                alpha_beta(
                    child,
                    depth - 1,
                    alpha,
                    beta,
                    board,
                    R,
                    C,
                    statistics,
                    heuristic_order
                )
            )

            alpha = max(alpha, value)

            if alpha >= beta:
                statistics["pruned"] += 1
                break

        return value

    else:

        value = math.inf

        for move, child, collected in children:

            value = min(
                value,
                alpha_beta(
                    child,
                    depth - 1,
                    alpha,
                    beta,
                    board,
                    R,
                    C,
                    statistics,
                    heuristic_order
                )
            )

            beta = min(beta, value)

            if alpha >= beta:
                statistics["pruned"] += 1
                break

        return value

## Cell 12: Find the Best Move Using Alpha-Beta

This function finds the best first move for MAX using Alpha-Beta pruning.

It can test:

- **Normal ordering:** UP → RIGHT → DOWN → LEFT
- **Heuristic ordering:** Energy-collecting moves first

In [12]:
def find_best_alpha_beta(
    initial_state,
    D,
    board,
    R,
    C,
    heuristic_order=False
):

    statistics = {
        "generated": 0,
        "expanded": 0,
        "pruned": 0
    }

    start_time = time.perf_counter()

    children = successors(
        initial_state,
        board,
        R,
        C,
        heuristic_order
    )

    statistics["generated"] += len(children)

    best_value = -math.inf
    best_move = None

    alpha = -math.inf
    beta = math.inf

    for move, child, collected in children:

        value = alpha_beta(
            child,
            D - 1,
            alpha,
            beta,
            board,
            R,
            C,
            statistics,
            heuristic_order
        )

        if value > best_value:
            best_value = value
            best_move = move

        alpha = max(alpha, best_value)

    execution_time = time.perf_counter() - start_time

    return (
        best_move,
        best_value,
        statistics["generated"],
        statistics["expanded"],
        statistics["pruned"],
        execution_time
    )

# Input and Execution

The following cells read the board, create the initial state, and run all required algorithms.

## Sample Input

```text
5 7
A.E.#..
.#..#E.
...#...
#E#..#.
....E.B
MAX
5
```

## Cell 13: Input

Enter the board exactly according to the required input format.

In [13]:
R, C = map(int, input().split())

board = []

for _ in range(R):
    row = input().strip()
    board.append(row)

start_player = input().strip()
D = int(input())

5 7
A.E.#..
.#..#E.
...#...
#E#..#.
....E.B
MAX
5


## Cell 14: Find Robot Positions and Energy Cells

This scans the board and stores:

- Position of MAX robot A
- Position of MIN robot B
- Positions of all remaining energy cells

In [14]:
max_pos = None
min_pos = None
energy = set()

for r in range(R):
    for c in range(C):

        if board[r][c] == 'A':
            max_pos = (r, c)

        elif board[r][c] == 'B':
            min_pos = (r, c)

        elif board[r][c] == 'E':
            energy.add((r, c))

## Cell 15: Create Initial State

The state representation used by this implementation is:

`(A_position, B_position, remaining_energy_cells, MAX_score, MIN_score, current_player)`

In [15]:
initial_state = (
    max_pos,
    min_pos,
    frozenset(energy),
    0,
    0,
    start_player
)

## Cell 16: Run Minimax

Run the standard depth-limited Minimax algorithm.

In [16]:
(
    minimax_move,
    minimax_value,
    minimax_generated,
    minimax_expanded,
    minimax_time
) = find_best_minimax(
    initial_state,
    D,
    board,
    R,
    C
)

## Cell 17: Display Minimax Results

This displays the best move and performance statistics for Minimax.

In [17]:
print("Algorithm: Minimax")
print("Best Move:", minimax_move)
print("Evaluation Score =", minimax_value)
print("Nodes Generated =", minimax_generated)
print("Nodes Expanded =", minimax_expanded)
print("Search Depth =", D)
print("Execution Time =", minimax_time)

Algorithm: Minimax
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 66
Search Depth = 5
Execution Time = 0.00026978200003213715


## Cell 18: Run Alpha-Beta with Normal Move Ordering

The move order is:

**UP → RIGHT → DOWN → LEFT**

In [18]:
(
    ab_move_normal,
    ab_value_normal,
    ab_generated_normal,
    ab_expanded_normal,
    ab_pruned_normal,
    ab_time_normal
) = find_best_alpha_beta(
    initial_state,
    D,
    board,
    R,
    C,
    heuristic_order=False
)

## Cell 19: Display Alpha-Beta Normal Results

This displays the performance of Alpha-Beta pruning using normal move ordering.

In [19]:
print("Algorithm: Alpha-Beta")
print("Move Ordering: Normal")
print("Best Move:", ab_move_normal)
print("Evaluation Score =", ab_value_normal)
print("Nodes Generated =", ab_generated_normal)
print("Nodes Expanded =", ab_expanded_normal)
print("Nodes Pruned =", ab_pruned_normal)
print("Search Depth =", D)
print("Execution Time =", ab_time_normal)

Algorithm: Alpha-Beta
Move Ordering: Normal
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 44
Nodes Expanded = 35
Nodes Pruned = 7
Search Depth = 5
Execution Time = 0.0002475330001061593


## Cell 20: Run Alpha-Beta with Heuristic Move Ordering

Moves that immediately collect an energy cell are tested first.

This ordering can increase the amount of Alpha-Beta pruning.

In [20]:
(
    ab_move_ordered,
    ab_value_ordered,
    ab_generated_ordered,
    ab_expanded_ordered,
    ab_pruned_ordered,
    ab_time_ordered
) = find_best_alpha_beta(
    initial_state,
    D,
    board,
    R,
    C,
    heuristic_order=True
)

## Cell 21: Display Alpha-Beta Heuristic Results

This displays the results using heuristic move ordering and allows comparison with normal ordering.

In [21]:
print("Algorithm: Alpha-Beta")
print("Move Ordering: Heuristic (Energy First)")
print("Best Move:", ab_move_ordered)
print("Evaluation Score =", ab_value_ordered)
print("Nodes Generated =", ab_generated_ordered)
print("Nodes Expanded =", ab_expanded_ordered)
print("Nodes Pruned =", ab_pruned_ordered)
print("Search Depth =", D)
print("Execution Time =", ab_time_ordered)

Algorithm: Alpha-Beta
Move Ordering: Heuristic (Energy First)
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 44
Nodes Expanded = 35
Nodes Pruned = 7
Search Depth = 5
Execution Time = 0.00018402099999548227
